In [1]:
# Install PyTorch (clean, stable, compatible with Python 3.12)
!pip uninstall -y torch torchvision torchaudio > /dev/null 2>&1
!pip install -q torch torchvision torchaudio

# Import and verify versions properly
import torch
import torchvision
from packaging import version

# Proper version checks (no fragile string splitting)
assert version.parse(torch.__version__) >= version.parse("1.12.0"), "torch>=1.12 required"
assert version.parse(torchvision.__version__) >= version.parse("0.13.0"), "torchvision>=0.13 required"

print(f"torch version: {torch.__version__}")
print(f"torchvision version: {torchvision.__version__}")
print(f"device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 112.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 70.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/59.5 MB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.9/

In [2]:
# Continue with regular imports
import matplotlib.pyplot as plt
import torch
import torchvision

from torch import nn
from torchvision import transforms

# Try to get torchinfo, install it if it doesn't work
try:
    from torchinfo import summary
except:
    print("[INFO] Couldn't find torchinfo... installing it.")
    !pip install -q torchinfo
    from torchinfo import summary

# Try to import the going_modular directory, download it from GitHub if it doesn't work
try:
    from going_modular.going_modular import data_setup, engine
    from helper_functions import download_data, set_seeds, plot_loss_curves
except:
    # Get the going_modular scripts
    print("[INFO] Couldn't find going_modular or helper_functions scripts... downloading them from GitHub.")
    !git clone https://github.com/mrdbourke/pytorch-deep-learning
    !mv pytorch-deep-learning/going_modular .
    !mv pytorch-deep-learning/helper_functions.py . # get the helper_functions.py script
    !rm -rf pytorch-deep-learning
    from going_modular.going_modular import data_setup, engine
    from helper_functions import download_data, set_seeds, plot_loss_curves

[INFO] Couldn't find torchinfo... installing it.
[INFO] Couldn't find going_modular or helper_functions scripts... downloading them from GitHub.
Cloning into 'pytorch-deep-learning'...
remote: Enumerating objects: 4410, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 4410 (delta 5), reused 2 (delta 2), pack-reused 4400 (from 2)
Receiving objects: 100% (4410/4410), 764.18 MiB | 30.63 MiB/s, done.
Resolving deltas: 100% (2661/2661), done.
Updating files: 100% (248/248), done.


In [3]:
# Download pizza, steak, sushi images from GitHub
data_20_percent_path = download_data(source="https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi_20_percent.zip",
                                     destination="pizza_steak_sushi_20_percent")

data_20_percent_path

[INFO] Did not find data/pizza_steak_sushi_20_percent directory, creating one...
[INFO] Downloading pizza_steak_sushi_20_percent.zip from https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi_20_percent.zip...
[INFO] Unzipping pizza_steak_sushi_20_percent.zip data...


PosixPath('data/pizza_steak_sushi_20_percent')

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [5]:
# Setup directory paths to train and test images
train_dir = data_20_percent_path / "train"
test_dir = data_20_percent_path / "test"

In [6]:
weights_B2 = torchvision.models.EfficientNet_B2_Weights.DEFAULT
transforms_B2 = weights_B2.transforms()
effnetb2 = torchvision.models.efficientnet_b2(weights=weights_B2).to(device)
for param in effnetb2.parameters():
    param.requires_grad = False

effnetb2.classifier = nn.Sequential(
    nn.Dropout(p=0.3, inplace=True),
    nn.Linear(in_features=1408, out_features=3, bias=True)
)

Downloading: "https://download.pytorch.org/models/efficientnet_b2_rwightman-c35c1473.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b2_rwightman-c35c1473.pth


100%|██████████| 35.2M/35.2M [00:00<00:00, 166MB/s]


In [7]:
from torchinfo import summary
summary(effnetb2, input_size=(1, 3, 224, 224),col_names=['trainable'])

Layer (type:depth-idx)                                  Trainable
EfficientNet                                            Partial
├─Sequential: 1-1                                       False
│    └─Conv2dNormActivation: 2-1                        False
│    │    └─Conv2d: 3-1                                 False
│    │    └─BatchNorm2d: 3-2                            False
│    │    └─SiLU: 3-3                                   --
│    └─Sequential: 2-2                                  False
│    │    └─MBConv: 3-4                                 False
│    │    └─MBConv: 3-5                                 False
│    └─Sequential: 2-3                                  False
│    │    └─MBConv: 3-6                                 False
│    │    └─MBConv: 3-7                                 False
│    │    └─MBConv: 3-8                                 False
│    └─Sequential: 2-4                                  False
│    │    └─MBConv: 3-9                                 False
│    

In [8]:
#model feature extractor
def create_effnetb2_model(num_classes=3,
                          seed = 32):
  weights = torchvision.models.EfficientNet_B2_Weights.DEFAULT
  transforms = weights.transforms()
  model = torchvision.models.efficientnet_b2(weights=weights).to(device)

  for param in effnetb2.parameters():
    param.requires_grad = False

  torch.manual_seed(seed)
  effnetb2.classifier = nn.Sequential(
      nn.Dropout(p=0.3, inplace=True),
      nn.Linear(in_features=1408, out_features=num_classes, bias=True)
  )
  return model, transforms


In [9]:
effnetb2, effnetb2_transforms = create_effnetb2_model()

In [10]:
effnetb2_transforms

ImageClassification(
    crop_size=[288]
    resize_size=[288]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BICUBIC
)

In [11]:
from going_modular.going_modular import data_setup

In [12]:
train_dataloader_effnet, test_dataloader_effnet, class_names = data_setup.create_dataloaders(
    train_dir=train_dir,
    test_dir=test_dir,
    transform=effnetb2_transforms,
    batch_size = 32
)

In [13]:
class_names

['pizza', 'steak', 'sushi']

In [14]:
from going_modular.going_modular import engine


In [15]:
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=effnetb2.parameters(), lr=0.001)
effnet_results = engine.train(model=effnetb2,
                              train_dataloader = train_dataloader_effnet,
                              test_dataloader = test_dataloader_effnet,
                              optimizer = optimizer,
                              loss_fn = loss_fn,
                              epochs = 10,
                              device=device
                              )

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch: 1 | train_loss: 1.9278 | train_acc: 0.6917 | test_loss: 0.3190 | test_acc: 0.9193
Epoch: 2 | train_loss: 0.2567 | train_acc: 0.9458 | test_loss: 0.2132 | test_acc: 0.9290
Epoch: 3 | train_loss: 0.4468 | train_acc: 0.8792 | test_loss: 0.1760 | test_acc: 0.9506
Epoch: 4 | train_loss: 0.1448 | train_acc: 0.9583 | test_loss: 0.1591 | test_acc: 0.9597
Epoch: 5 | train_loss: 0.0519 | train_acc: 0.9625 | test_loss: 0.1326 | test_acc: 0.9659
Epoch: 6 | train_loss: 0.1972 | train_acc: 0.9479 | test_loss: 0.1951 | test_acc: 0.9062
Epoch: 7 | train_loss: 0.0897 | train_acc: 0.9792 | test_loss: 0.1607 | test_acc: 0.9534
Epoch: 8 | train_loss: 0.0568 | train_acc: 0.9833 | test_loss: 0.1022 | test_acc: 0.9722
Epoch: 9 | train_loss: 0.0991 | train_acc: 0.9646 | test_loss: 0.1101 | test_acc: 0.9722
Epoch: 10 | train_loss: 0.2575 | train_acc: 0.9437 | test_loss: 0.3015 | test_acc: 0.9040


In [16]:
from going_modular.going_modular import utils

utils.save_model(model=effnetb2,
                 target_dir='models',
                 model_name= "effnet_reature_extractor.pth")

[INFO] Saving model to: models/effnet_reature_extractor.pth


In [17]:
from pathlib import Path

effnetb2_size = Path("models/effnet_reature_extractor.pth").stat().st_size / (1024 * 1024)
effnetb2_size

35.19297122955322

In [18]:
effnetb2_params = sum(p.numel() for p in effnetb2.parameters())
effnetb2_params

9109994

In [19]:
# Create a dictionary with EffNetB2 statistics
effnetb2_stats = {"test_loss": effnet_results["test_loss"][-1],
                  "test_acc": effnet_results["test_acc"][-1],
                  "number_of_parameters": effnetb2_params,
                  "model_size (MB)": effnetb2_size}
effnetb2_stats

{'test_loss': 0.30147661492228506,
 'test_acc': 0.9039772727272727,
 'number_of_parameters': 9109994,
 'model_size (MB)': 35.19297122955322}

In [20]:
#model feature extractor
def create_vit_model(num_classes=3,
                          seed = 32):
  weights = torchvision.models.ViT_B_16_Weights.DEFAULT
  transforms = weights.transforms()
  model = torchvision.models.vit_b_16(weights=weights).to(device)

  for param in model.parameters():
    param.requires_grad = False

  torch.manual_seed(seed)
  model.heads = nn.Sequential(
      nn.Linear(in_features=768, out_features=num_classes, bias=True)
  )
  return model, transforms


In [21]:
vit, vit_transforms = create_vit_model()
vit_transforms

Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:01<00:00, 185MB/s]


ImageClassification(
    crop_size=[224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)

In [22]:
train_dataloader_vit, test_dataloader_vit, class_names = data_setup.create_dataloaders(
    train_dir=train_dir,
    test_dir=test_dir,
    transform=vit_transforms,
    batch_size = 32
)
len(train_dataloader_vit)

15

In [23]:
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=vit.parameters(), lr=1e-3)
vit_results = engine.train(model=vit,
                              train_dataloader = train_dataloader_vit,
                              test_dataloader = test_dataloader_vit,
                              optimizer = optimizer,
                              loss_fn = loss_fn,
                              epochs = 10,
                              device=device
                              )

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch: 1 | train_loss: 0.5511 | train_acc: 0.8250 | test_loss: 0.2228 | test_acc: 0.9722
Epoch: 2 | train_loss: 0.2426 | train_acc: 0.9187 | test_loss: 0.1201 | test_acc: 0.9875
Epoch: 3 | train_loss: 0.1490 | train_acc: 0.9646 | test_loss: 0.1030 | test_acc: 0.9875
Epoch: 4 | train_loss: 0.1185 | train_acc: 0.9667 | test_loss: 0.0881 | test_acc: 0.9875
Epoch: 5 | train_loss: 0.1023 | train_acc: 0.9688 | test_loss: 0.0776 | test_acc: 0.9875
Epoch: 6 | train_loss: 0.1789 | train_acc: 0.9417 | test_loss: 0.0718 | test_acc: 0.9875
Epoch: 7 | train_loss: 0.0801 | train_acc: 0.9854 | test_loss: 0.0811 | test_acc: 0.9784
Epoch: 8 | train_loss: 0.0816 | train_acc: 0.9917 | test_loss: 0.0675 | test_acc: 0.9875
Epoch: 9 | train_loss: 0.0639 | train_acc: 0.9938 | test_loss: 0.0571 | test_acc: 0.9875
Epoch: 10 | train_loss: 0.0530 | train_acc: 0.9938 | test_loss: 0.0537 | test_acc: 0.9875


In [24]:
from going_modular.going_modular import utils

utils.save_model(model=vit,
                 target_dir='models',
                 model_name= "vit_reature_extractor.pth")

[INFO] Saving model to: models/vit_reature_extractor.pth


In [25]:
from pathlib import Path

vit_size = Path("models/vit_reature_extractor.pth").stat().st_size / (1024 * 1024)
vit_size

327.3647336959839

In [26]:
vit_params = sum(p.numel() for p in vit.parameters())
vit_params

85800963

In [27]:
# Create a dictionary with EffNetB2 statistics
vit_stats = {"test_loss": vit_results["test_loss"][-1],
                  "test_acc": vit_results["test_acc"][-1],
                  "number_of_parameters": vit_params,
                  "model_size (MB)": vit_size}
vit_stats

{'test_loss': 0.053729654476046565,
 'test_acc': 0.9875,
 'number_of_parameters': 85800963,
 'model_size (MB)': 327.3647336959839}

In [28]:
from pathlib import Path

test_data_path = list(Path(test_dir).glob("*/*.jpg"))
test_data_path[0]

PosixPath('data/pizza_steak_sushi_20_percent/test/pizza/3375083.jpg')

In [29]:
from timeit import default_timer as timer
from tqdm.auto import tqdm
from typing import List, Dict
import pathlib
from pathlib import Path
from PIL import Image

def pred_and_store(paths: List[pathlib.Path],
                   model: torch.nn.Module,
                   transform: torchvision.transforms,
                   class_names: List[str],
                   device: torch.device = "cuda" if torch.cuda.is_available() else 'cpu'
                   ):
  pred_list = []

  for path in tqdm(paths):

    pred_dict = {}

    pred_dict["image_path"] = path
    class_name = path.parent.stem
    pred_dict["actual_class"] = class_name

    start_time = timer()

    image = Image.open(path)

    transformed_image = transform(image).unsqueeze(dim=0).to(device)

    model = model.to(device)
    model.eval()

    with torch.inference_mode():
      pred_logits = model(transformed_image)
      pred_prob = torch.softmax(pred_logits, dim=1)
      pred_label = torch.argmax(pred_prob)
      pred_dict["predicted_class"] = class_names[pred_label.cpu()]
      pred_dict["predicted_probability"] = torch.max(pred_prob.cpu())

      end_time = timer()
      pred_dict["inference_time"] = round(end_time - start_time, 3)
      pred_dict['correct'] = class_name == class_names[pred_label.cpu()]

      pred_list.append(pred_dict)
  return pred_list




In [30]:
effnet_preds = pred_and_store(paths=test_data_path,
                              model=effnetb2,
                              transform=effnetb2_transforms,
                              class_names=class_names,
                              device='cpu')

  0%|          | 0/150 [00:00<?, ?it/s]

In [31]:
effnet_preds[:2]

[{'image_path': PosixPath('data/pizza_steak_sushi_20_percent/test/pizza/3375083.jpg'),
  'actual_class': 'pizza',
  'predicted_class': 'pizza',
  'predicted_probability': tensor(0.9231),
  'inference_time': 0.175,
  'correct': True},
 {'image_path': PosixPath('data/pizza_steak_sushi_20_percent/test/pizza/3486640.jpg'),
  'actual_class': 'pizza',
  'predicted_class': 'pizza',
  'predicted_probability': tensor(0.9832),
  'inference_time': 0.101,
  'correct': True}]

In [32]:
import pandas as pd
effned_df = pd.DataFrame(effnet_preds)
effned_df.head()

,image_path,actual_class,predicted_class,predicted_probability,inference_time,correct
0,data/pizza_steak_sushi_20_percent/test/pizza/3...,pizza,pizza,tensor(0.9231),0.175,True
1,data/pizza_steak_sushi_20_percent/test/pizza/3...,pizza,pizza,tensor(0.9832),0.101,True
2,data/pizza_steak_sushi_20_percent/test/pizza/2...,pizza,pizza,tensor(0.9965),0.095,True
3,data/pizza_steak_sushi_20_percent/test/pizza/3...,pizza,pizza,tensor(0.8209),0.088,True
4,data/pizza_steak_sushi_20_percent/test/pizza/3...,pizza,pizza,tensor(0.9996),0.106,True


In [33]:
effned_df.correct.value_counts()


,count
correct,
True,136
False,14


In [34]:
eff_time = effned_df['inference_time'].mean().round(4)
eff_time

np.float64(0.0949)

In [35]:
effnetb2_stats['average inference time'] = eff_time

In [36]:
vit_preds = pred_and_store(paths=test_data_path,
                              model=vit,
                              transform=vit_transforms,
                              class_names=class_names,
                              device='cpu') #some devices don't have gpu

  0%|          | 0/150 [00:00<?, ?it/s]

In [37]:
vit_df = pd.DataFrame(vit_preds)
vit_df.head()

,image_path,actual_class,predicted_class,predicted_probability,inference_time,correct
0,data/pizza_steak_sushi_20_percent/test/pizza/3...,pizza,pizza,tensor(0.9983),0.549,True
1,data/pizza_steak_sushi_20_percent/test/pizza/3...,pizza,pizza,tensor(0.9956),0.378,True
2,data/pizza_steak_sushi_20_percent/test/pizza/2...,pizza,pizza,tensor(0.9986),0.386,True
3,data/pizza_steak_sushi_20_percent/test/pizza/3...,pizza,pizza,tensor(0.9977),0.368,True
4,data/pizza_steak_sushi_20_percent/test/pizza/3...,pizza,pizza,tensor(0.9982),0.362,True


In [38]:
vit_df.correct.value_counts()


,count
correct,
True,148
False,2


In [39]:
vit_time = vit_df['inference_time'].mean().round(4)

In [40]:
vit_stats['average inference time'] = vit_time

In [41]:
df = pd.DataFrame(data=[effnetb2_stats, vit_stats])
df['model'] = ['effnet', 'vit']
df.set_index('model', inplace=True)
df

,test_loss,test_acc,number_of_parameters,model_size (MB),average inference time
model,,,,,
effnet,0.301477,0.903977,9109994,35.192971,0.0949
vit,0.053730,0.987500,85800963,327.364734,0.4611


In [42]:
try:
  import gradio as gr

except:
  !pip -q install gradio
  import gradio as gr

In [43]:
effnetb2.to('cpu')
next(iter(effnetb2.parameters())).device

device(type='cpu')

In [44]:
class_names

['pizza', 'steak', 'sushi']

In [45]:
from typing import Tuple, Dict

def predict(img, model, transform) -> Tuple[Dict, float]:
  start_time = timer()
  img = transform(img).unsqueeze(dim=0)
  model.eval()
  with torch.inference_mode():
    pred_logits = model(img)
    pred_prob = torch.softmax(pred_logits, dim=1)

  pred_labels_and_prob = {class_names[i]: float(pred_prob[0][i]) for i in range(len(class_names))}
  end_timer = timer()
  pred_time = round(end_timer - start_time, 5)

  return pred_labels_and_prob, pred_time

In [46]:
predict(img=Image.open(test_data_path[0]),
        model=effnetb2, transform=effnetb2_transforms)

({'pizza': 0.9230570793151855,
  'steak': 0.011378363706171513,
  'sushi': 0.00013146224955562502},
 0.10622)

In [47]:
example_list = [[str(path)] for path in test_data_path[:3]]
example_list

[['data/pizza_steak_sushi_20_percent/test/pizza/3375083.jpg'],
 ['data/pizza_steak_sushi_20_percent/test/pizza/3486640.jpg'],
 ['data/pizza_steak_sushi_20_percent/test/pizza/2782998.jpg']]

In [48]:
# import gradio as gr
# title = "Food vision"
# description = "Model predictS three classes: pizza, steak, sushi"
# # Create a small wrapper that "pre-fills" your model and transforms
# def gradio_predict(img):
#     return predict(img, model=effnetb2, transform=effnetb2_transforms)

# demo = gr.Interface(
#     fn=gradio_predict, # Use the wrapper here
#     inputs=gr.Image(type='pil'),
#     outputs=[
#         gr.Label(num_top_classes=3, label='Predictions'),
#         gr.Number(label='Prediction time (s)')
#     ],
#     examples=example_list,
#     title=title,
#     description=description
# )

# demo.launch(share=True)

In [53]:
import shutil
from pathlib import Path

foodvision_mini_demo_path = Path("demos/foodvision_mini_demo")

if foodvision_mini_demo_path.exists():
  shutil.rmtree(foodvision_mini_demo_path)
  foodvision_mini_demo_path.mkdir(parents=True, exist_ok=True)

else:

  foodvision_mini_demo_path.mkdir(parents=True, exist_ok=True)


In [54]:
foodvision_mini_demo_path

PosixPath('demos/foodvision_mini_demo')

In [55]:
foodvision_mini_examples = foodvision_mini_demo_path / "examples"
foodvision_mini_examples.mkdir(parents=True, exist_ok=True)

In [60]:
for example in (example_list):
  print(example[0])
  shutil.copy(example[0], foodvision_mini_examples)

data/pizza_steak_sushi_20_percent/test/pizza/3375083.jpg
data/pizza_steak_sushi_20_percent/test/pizza/3486640.jpg
data/pizza_steak_sushi_20_percent/test/pizza/2782998.jpg


In [87]:
import os
example_list = [os.path.join('/content/demos/foodvision_mini_demo/examples', example) for example in os.listdir('/content/demos/foodvision_mini_demo/examples')]
example_list

['/content/demos/foodvision_mini_demo/examples/3375083.jpg',
 '/content/demos/foodvision_mini_demo/examples/3486640.jpg',
 '/content/demos/foodvision_mini_demo/examples/2782998.jpg']

In [79]:
effnet_foodvision_path = "models/effnet_reature_extractor.pth"
effnet_foodvision_destination = foodvision_mini_demo_path/"effnet_foodvision.pth"
shutil.copy(effnet_foodvision_path, effnet_foodvision_destination)

PosixPath('demos/foodvision_mini_demo/effnet_foodvision.pth')

In [80]:
%%writefile demos/foodvision_mini_demo/model.py
import torch
import torchvision
from typing import Tuple, Dict
from torch import nn

#model feature extractor
def create_effnetb2_model(num_classes=3,
                          seed = 32):
  weights = torchvision.models.EfficientNet_B2_Weights.DEFAULT
  transforms = weights.transforms()
  model = torchvision.models.efficientnet_b2(weights=weights)

  for param in model.parameters():
    param.requires_grad = False

  torch.manual_seed(seed)
  model.classifier = nn.Sequential(
      nn.Dropout(p=0.3, inplace=True),
      nn.Linear(in_features=1408, out_features=num_classes, bias=True)
  )
  return model, transforms


Overwriting demos/foodvision_mini_demo/model.py


In [81]:
from demos.foodvision_mini_demo import model
effnet_model , effnet_trans = model.create_effnetb2_model()

In [82]:
effnet_trans

ImageClassification(
    crop_size=[288]
    resize_size=[288]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BICUBIC
)

In [103]:
%%writefile demos/foodvision_mini_demo/app.py

import gradio as gr
import os
import torch
import model
from typing import Tuple, Dict
from PIL import Image
from timeit import default_timer as timer
from tqdm.auto import tqdm

class_names = ['pizza', 'steak', 'sushi']
effnet_model , effnet_trans = model.create_effnetb2_model()
effnet_model.load_state_dict(
    torch.load(
        f="effnet_foodvision.pth",
        map_location=torch.device("cpu")
    )
)
title = "Food vision"
description = "Model predictS three classes: pizza, steak, sushi"

def predict(img, model, transform) -> Tuple[Dict, float]:
  start_time = timer()
  img = transform(img).unsqueeze(dim=0)
  model.eval()
  with torch.inference_mode():
    pred_logits = model(img)
    pred_prob = torch.softmax(pred_logits, dim=1)

  pred_labels_and_prob = {class_names[i]: float(pred_prob[0][i]) for i in range(len(class_names))}
  end_timer = timer()
  pred_time = round(end_timer - start_time, 5)

  return pred_labels_and_prob, pred_time

def gradio_predict(img):
    return predict(img, model=effnet_model, transform=effnet_trans)

example_list = [[os.path.join("examples", example)] for example in os.listdir("examples")]


demo = gr.Interface(
    fn=gradio_predict, # Use the wrapper here
    inputs=gr.Image(type='pil'),
    outputs=[
        gr.Label(num_top_classes=3, label='Predictions'),
        gr.Number(label='Prediction time (s)')
    ],
    examples=example_list,
    title=title,
    description=description
)

demo.launch(share=True)


Overwriting demos/foodvision_mini_demo/app.py


In [104]:
%%writefile demos/foodvision_mini_demo/requirements.txt
torch==2.11
gradio==5.50



Overwriting demos/foodvision_mini_demo/requirements.txt


In [105]:
gr.__version__

'5.50.0'

In [106]:
print("s")

s


In [107]:
!ls demos/foodvision_mini_demo

app.py		       examples  __pycache__
effnet_foodvision.pth  model.py  requirements.txt


In [108]:
!cd demos/foodvision_mini_demo && zip -r ../foodvision_mini.zip *

  adding: app.py (deflated 52%)
  adding: effnet_foodvision.pth (deflated 8%)
  adding: examples/ (stored 0%)
  adding: examples/3375083.jpg (deflated 0%)
  adding: examples/3486640.jpg (deflated 0%)
  adding: examples/2782998.jpg (deflated 1%)
  adding: model.py (deflated 46%)
  adding: __pycache__/ (stored 0%)
  adding: __pycache__/model.cpython-312.pyc (deflated 37%)
  adding: requirements.txt (stored 0%)
